# LoopMind — Full Training (CNN + Transformer)
**Run location**: browser Colab (colab.research.google.com), T4 GPU  
**Data**: Google Drive → `MyDrive/LoopMind/slakh2100/`  
**Outputs**: Drive → `MyDrive/LoopMind/cache/` and `MyDrive/LoopMind/checkpoints/`

Run cells top to bottom. If session disconnects, re-run Cell 1–3, then Cell 5 with `--resume`.

In [ ]:
# ── Cell 0: Download Slakh2100 to Drive (run ONCE only) ───────────────────────
#
# Strategy: stream download → pipe directly into tar (no archive saved to Drive)
#           then delete tracks beyond our 400-track subset.
# Drive space needed: ~10 GB total (400/1500 tracks of the train split).
#
# Before running:
#   1. Go to https://zenodo.org/record/4599666
#   2. Find 'slakh2100_flac_redux.tar.gz', right-click → Copy link address
#   3. Paste it as ZENODO_URL below

from google.colab import drive
drive.mount('/content/drive')

import os, shutil

DRIVE_ROOT = '/content/drive/MyDrive/LoopMind'
SLAKH_DIR  = f'{DRIVE_ROOT}/slakh2100'
MAX_TRACKS = 400   # keep only this many tracks from the train split

# ── Paste the Zenodo direct download URL here ──────────────────────────────────
ZENODO_URL = ''
# ──────────────────────────────────────────────────────────────────────────────

if os.path.isdir(SLAKH_DIR):
    print('slakh2100/ already exists on Drive, skipping download.')
else:
    assert ZENODO_URL, 'Set ZENODO_URL above before running this cell!'
    os.makedirs(DRIVE_ROOT, exist_ok=True)

    # Stream download and extract directly to Drive — no archive file saved
    print('Streaming download + extraction (no archive stored) ...')
    print('This takes ~1-2 hours. Keep the browser tab active.')
    !wget -qO- "{ZENODO_URL}" | tar -xz -C "{DRIVE_ROOT}"
    print('Extraction complete.')

    # Keep only the first MAX_TRACKS tracks from train split to save Drive space
    train_dir = os.path.join(SLAKH_DIR, 'train')
    all_tracks = sorted([
        d for d in os.listdir(train_dir)
        if os.path.isdir(os.path.join(train_dir, d))
    ])
    to_delete = all_tracks[MAX_TRACKS:]
    if to_delete:
        print(f'Deleting {len(to_delete)} extra train tracks to save space ...')
        for t in to_delete:
            shutil.rmtree(os.path.join(train_dir, t))
        print('Done.')

    # Report final state
    for split in ['train', 'validation']:
        d = os.path.join(SLAKH_DIR, split)
        n = len(os.listdir(d)) if os.path.isdir(d) else 0
        print(f'  {split}: {n} tracks kept')

In [ ]:
# ── Cell 1: Mount Drive + clone / update repo ─────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/LoopMind'

if not os.path.exists(REPO):
    !git clone https://github.com/ranxindeng-blip/LoopMind {REPO}
else:
    !cd {REPO} && git pull

DATA_ROOT    = '/content/drive/MyDrive/LoopMind/slakh2100_flac_redux'
CACHE_DIR    = '/content/drive/MyDrive/LoopMind/cache'
CKPT_DIR     = '/content/drive/MyDrive/LoopMind/checkpoints'
LIBRARY_PATH = '/content/drive/MyDrive/LoopMind/cache/library.pt'

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,  exist_ok=True)
print('Drive mounted. Paths ready.')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q pretty_midi librosa soundfile gradio pydub pyyaml
print('Dependencies installed.')

In [ ]:
# ── Cell 3: Verify Slakh2100 data structure ───────────────────────────────────
import os

for split in ['train', 'validation']:
    split_dir = os.path.join(DATA_ROOT, split)
    if os.path.isdir(split_dir):
        tracks = [d for d in os.listdir(split_dir)
                  if os.path.isdir(os.path.join(split_dir, d))]
        print(f'{split}: {len(tracks)} tracks')
    else:
        print(f'WARNING: {split_dir} not found!')

# Peek at one track to confirm audio format (.flac or .wav)
train_dir    = os.path.join(DATA_ROOT, 'train')
sample_track = os.path.join(train_dir, sorted(os.listdir(train_dir))[0])
print(f'\nSample track: {os.path.basename(sample_track)}')
print('  MIDI:  ', os.listdir(os.path.join(sample_track, 'MIDI'))[:4])
print('  stems: ', os.listdir(os.path.join(sample_track, 'stems'))[:4])

In [ ]:
# ── Cell 4: Extract pairs + features (run once, cached to Drive) ──────────────
# ~20-40 min for 400 tracks. Already-cached .npy files are skipped on re-run.

import sys
sys.path.insert(0, REPO)

from data.extract_pairs import extract_pairs
from data.features import extract_and_cache

MAX_TRACKS = 400

print(f'Scanning {MAX_TRACKS} tracks ...')
records = extract_pairs(DATA_ROOT, CACHE_DIR, max_tracks=MAX_TRACKS)
print(f'\nTotal records: {len(records)}')

print('\nExtracting features (piano roll + mel) ...')
features = extract_and_cache(records, CACHE_DIR)
print('\nFeature extraction complete.')

In [ ]:
# ── Cell 5: Train ─────────────────────────────────────────────────────────────
# First run: as-is.
# After disconnect: uncomment --resume to continue from last.pt.

!cd {REPO} && python train.py \
    --data_root   {DATA_ROOT} \
    --cache_dir   {CACHE_DIR} \
    --ckpt_dir    {CKPT_DIR} \
    --max_tracks  400 \
    --epochs      100 \
    --batch_size  64 \
    --lr          1e-4 \
    --hidden      256 \
    --embed_dim   128 \
    --n_heads     4 \
    --n_layers    2
    # --resume

In [ ]:
# ── Cell 6: Evaluate ──────────────────────────────────────────────────────────
!cd {REPO} && python evaluate.py \
    --ckpt_path  {CKPT_DIR}/best.pt \
    --data_root  {DATA_ROOT} \
    --cache_dir  {CACHE_DIR} \
    --max_tracks 400

In [ ]:
# ── Cell 7: Build stem library ────────────────────────────────────────────────
!cd {REPO} && python build_library.py \
    --data_root    {DATA_ROOT} \
    --cache_dir    {CACHE_DIR} \
    --ckpt_path    {CKPT_DIR}/best.pt \
    --max_tracks   400 \
    --library_path {LIBRARY_PATH}

In [ ]:
# ── Cell 8: Launch Gradio demo (full model) ───────────────────────────────────
import sys, torch
sys.path.insert(0, REPO)

from models.dual_encoder import DualEncoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt   = torch.load(f'{CKPT_DIR}/best.pt', map_location=device, weights_only=False)
args   = ckpt['args']
model  = DualEncoder(
    hidden    = args['hidden'],
    embed_dim = args['embed_dim'],
    n_heads   = args.get('n_heads', 4),
    n_layers  = args.get('n_layers', 2),
)
model.load_state_dict(ckpt['model'])

library = torch.load(LIBRARY_PATH, map_location='cpu', weights_only=False)

from demo.app import launch
launch(model, library, device=device, share=True)